# People represented by address samples

USPS-derived address points carry census geography (`statefp`, `countyfp`, `tractce`, `blkgrpce`). We join **ACS 5-year** block-group total population (table **B01003**) and derive a **population weight** for each sampled address.

Within a block group, let \(n\) be the number of sampled addresses and \(P\) the census population. **Addresses per person** is \(n/P\). Each address is treated as standing in for an equal share of the block group, so the weight that reproduces \(P\) when you sum over addresses is **people per address** \(P/n = 1 / (n/P)\). That is what we store as `population_weight`.

In [1]:
from pathlib import Path

import pandas as pd
import requests

DATA_DIR = Path("../data")
ADDRESSES_DIR = DATA_DIR / "addresses"

# Five VT counties: Caledonia, Essex, Lamoille, Orleans, Washington (3-digit county FIPS)
TARGET_COUNTYFP = {"005", "009", "015", "019", "023"}

ACS_YEAR = 2022
ACS_DATASET = "acs/acs5"
CENSUS_BASE = f"https://api.census.gov/data/{ACS_YEAR}/{ACS_DATASET}"

PCT100_DIR = ADDRESSES_DIR / "100pct"
address_files = sorted(PCT100_DIR.glob("*-addresses.csv"))
assert len(address_files) == 5, f"Expected 5 CSVs in {PCT100_DIR}, found {len(address_files)}"
addresses = pd.concat([pd.read_csv(f) for f in address_files], ignore_index=True)
addresses = addresses.rename(columns={"latitude": "lat", "longitude": "lon"})
addresses = addresses.dropna(subset=["lat", "lon", "statefp", "countyfp", "tractce", "blkgrpce"])
addresses["countyfp"] = addresses["countyfp"].astype(int)
addresses = addresses[addresses["countyfp"].astype(str).str.zfill(3).isin(TARGET_COUNTYFP)].copy()


def block_group_geoid(row: pd.Series) -> str:
    """12-digit block-group GEOID: SS CCC TTTTTT G."""
    st = f"{int(row['statefp']):02d}"
    co = f"{int(row['countyfp']):03d}"
    tr = str(row["tractce"]).split(".")[0].zfill(6)
    bg = str(int(float(row["blkgrpce"])))
    return f"{st}{co}{tr}{bg}"


addresses["geoid_bg"] = addresses.apply(block_group_geoid, axis=1)
print(f"{len(addresses)} addresses in target counties with block-group GEOIDs")
addresses[["address_full", "city", "geoid_bg"]].head()

86672 addresses in target counties with block-group GEOIDs


,address_full,city,geoid_bg
0,"190 STEALTH RIDGE LN SAINT JOHNSBURY, VT 05819",SAINT JOHNSBURY,500059575001
1,"296 WILLSON RD DANVILLE, VT 05819",DANVILLE,500059576002
2,"156 PAPERMILL RD RYEGATE, VT 05042",RYEGATE,500059578003
3,"1066 RAILROAD ST SAINT JOHNSBURY, VT 05819",SAINT JOHNSBURY,500059574002
4,"1842 HARDWICK ST HARDWICK, VT 05836",HARDWICK,500059577001


## ACS population by block group

Pull **B01003_001E** (total population) for Vermont counties 005, 009 (Essex), 015, 019, and 023. The Census API does not require a key for light use; add `CENSUS_API_KEY` to `.env` if you hit rate limits.

In [2]:
import os

from dotenv import load_dotenv

load_dotenv(Path("../.env"))


def fetch_acs_block_group_pop(county_fp: str) -> pd.DataFrame:
    params = {
        "get": "NAME,B01003_001E",
        "for": "block group:*",
        "in": f"state:50 county:{county_fp}",
    }
    key = os.environ.get("CENSUS_API_KEY", "")
    if key:
        params["key"] = key
    r = requests.get(CENSUS_BASE, params=params, timeout=120)
    r.raise_for_status()
    rows = r.json()
    cols, *data = rows
    return pd.DataFrame(data, columns=cols)


parts = [fetch_acs_block_group_pop(c) for c in sorted(TARGET_COUNTYFP)]
bg_pop = pd.concat(parts, ignore_index=True)
bg_pop["population"] = pd.to_numeric(bg_pop["B01003_001E"], errors="coerce")
miss = bg_pop["population"] < 0
if miss.any():
    bg_pop.loc[miss, "population"] = pd.NA
bg_pop["geoid_bg"] = (
    bg_pop["state"].astype(str).str.zfill(2)
    + bg_pop["county"].astype(str).str.zfill(3)
    + bg_pop["tract"].astype(str).str.zfill(6)
    + bg_pop["block group"].astype(str)
)
bg_pop = bg_pop[["geoid_bg", "NAME", "population"]].drop_duplicates("geoid_bg")
print(f"{len(bg_pop)} block groups from ACS ({ACS_YEAR} ACS 5-year)")
bg_pop.head()

128 block groups from ACS (2022 ACS 5-year)


,geoid_bg,NAME,population
0,500059570001,Block Group 1; Census Tract 9570; Caledonia Co...,1475
1,500059570002,Block Group 2; Census Tract 9570; Caledonia Co...,1049
2,500059570003,Block Group 3; Census Tract 9570; Caledonia Co...,1603
3,500059571001,Block Group 1; Census Tract 9571; Caledonia Co...,653
4,500059571002,Block Group 2; Census Tract 9571; Caledonia Co...,1397


## Join and weights

For each block group appearing in the sample, count addresses `n`. Then `addresses_per_person = n / P` and `population_weight = P / n`.

In [3]:
merged = addresses.merge(bg_pop, on="geoid_bg", how="left", validate="m:1")
missing_pop = merged["population"].isna()
if missing_pop.any():
    print(f"Warning: {missing_pop.sum()} rows with no ACS match")
    print(merged.loc[missing_pop, "geoid_bg"].value_counts().head())

n_per_bg = merged.groupby("geoid_bg").size().rename("addresses_in_bg")
merged = merged.merge(n_per_bg, on="geoid_bg", how="left")

merged["addresses_per_person"] = merged["addresses_in_bg"] / merged["population"]
merged["population_weight"] = merged["population"] / merged["addresses_in_bg"]

bad = (merged["population"].isna()) | (merged["population"] <= 0) | (merged["addresses_in_bg"] <= 0)
merged.loc[bad, "addresses_per_person"] = pd.NA
merged.loc[bad, "population_weight"] = pd.NA

chk = merged.dropna(subset=["population_weight"]).groupby("geoid_bg").agg(
    pop=("population", "first"),
    wsum=("population_weight", "sum"),
)
chk["diff"] = (chk["wsum"] - chk["pop"]).abs()
assert (chk["diff"] < 0.01).all(), "Weights should sum to block-group population"

print(
    f"population_weight sum over all rows ≈ {merged['population_weight'].sum():,.0f} "
    "(population only in sampled block groups)"
)
merged[[
    "address_full",
    "city",
    "geoid_bg",
    "population",
    "addresses_in_bg",
    "addresses_per_person",
    "population_weight",
]].head(10)

population_weight sum over all rows ≈ 149,598 (population only in sampled block groups)


,address_full,city,geoid_bg,population,addresses_in_bg,addresses_per_person,population_weight
0,"190 STEALTH RIDGE LN SAINT JOHNSBURY, VT 05819",SAINT JOHNSBURY,500059575001,1444,876,0.606648,1.648402
1,"296 WILLSON RD DANVILLE, VT 05819",DANVILLE,500059576002,1038,637,0.613680,1.629513
2,"156 PAPERMILL RD RYEGATE, VT 05042",RYEGATE,500059578003,1103,734,0.665458,1.502725
3,"1066 RAILROAD ST SAINT JOHNSBURY, VT 05819",SAINT JOHNSBURY,500059574002,814,432,0.530713,1.884259
4,"1842 HARDWICK ST HARDWICK, VT 05836",HARDWICK,500059577001,1729,888,0.513592,1.947072
5,"862 DUTTON RD HARDWICK, VT 05843",HARDWICK,500059577002,1221,685,0.561016,1.782482
6,"76 WATERMAN CIR SAINT JOHNSBURY, VT 05819",SAINT JOHNSBURY,500059575001,1444,876,0.606648,1.648402
7,"4118 DARLING HILL RD BURKE, VT 05832",BURKE,500059571002,1397,859,0.614889,1.626310
8,"632 S RIDGE RD SUTTON, VT 05867",SUTTON,500059570001,1475,1249,0.846780,1.180945
9,"7 PENNY LN STANNARD, VT 05842",STANNARD,500059570003,1603,1328,0.828447,1.207078


In [4]:
out_path = DATA_DIR / "addresses_with_population_weights.csv"
merged.to_csv(out_path, index=False)
print(f"Wrote {len(merged)} rows to {out_path}")

Wrote 86672 rows to ../data/addresses_with_population_weights.csv
